In [27]:
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords
import spacy
import locale
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

### Texto crudo

In [28]:
path = "./corpora/excelsior/"
filename = "e990505_mod.htm"

with open(path + filename, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:1000])

<!DOCTYPE html>
<html lang='es'>
<head>
<meta charset='UTF-8'>
<title>e990505_mod.htm</title>
</head>
<body>

http://www.excelsior.com.mx/9905/990505/art01.html


<title>Excelsior: Editorial</title>


<basefont size="3">
<img src="/gifs/ban_editorial.gif">
<hr><h5 align="right">Miércoles 05 de Mayo de 1999
</h5>
<h3>Epigrama
</h3><p></p><p>
<b>JORGE MANSILLA TORRES</b></p><p>
</p><p>
La diputada Raquel Sevilla, dueña de una violencia verbal envidiable, camina con un collarín ortopédico.</p><p>
Y no faltó el periodista</p><p>
que con lenguaje letal,</p><p>
dijo que la asambleísta</p><p>
¡mejor se ponga un bozal..!</p><p>
</p><hr><center><font size="-1">
<a href="editorial.html">Editorial</a> -
<a href="art02.html">Nota siguiente</a>
</font></center>
http://www.excelsior.com.mx/9905/990505/art02.html


<title>Excelsior: Editorial</title>


<basefont size="3">
<img src="/gifs/ban_editorial.gif">
<hr><h5 align="right">Miércoles 05 de Mayo de 1999
</h5>
<h3>Liquidar a Milosevic
</h3><p></p>

### Ingeniería de Características Lingüisticas

Eliminar etiquetas html.

Antes de extraer el texto se quita la plantilla del periódico, que se repite en cada nota y no es contenido:
- `<title>`: el nombre del archivo y los encabezados "Excelsior: ..."
- `<h4>` y `<h5>`: la fecha
- `<font>`: la navegación del pie ("Nacionales - Nota siguiente")

In [29]:
soup = BeautifulSoup(raw_text, "html.parser")
for tag in soup.find_all(["title", "h4", "h5", "font"]):
    tag.decompose()
tmp = soup.get_text(separator="\n", strip=True)
print(tmp[:1000])

http://www.excelsior.com.mx/9905/990505/art01.html
Epigrama
JORGE MANSILLA TORRES
La diputada Raquel Sevilla, dueña de una violencia verbal envidiable, camina con un collarín ortopédico.
Y no faltó el periodista
que con lenguaje letal,
dijo que la asambleísta
¡mejor se ponga un bozal..!
http://www.excelsior.com.mx/9905/990505/art02.html
Liquidar a Milosevic
Tranquilizar la Mala Conciencia
JORGE RUIZ DUEÑAS
LOS voceros de la Casa Blanca y de la OTAN se han referido, a la fórmula de los bombardeos sobre las ciudades yugoslavas, como el castigo necesario para liquidar a Slobodan Milosevic. Incluso, se dice, el conflicto actual es el precio de no haber actuado antes en el desmembramiento de la Yugoslavia de Tito. El sufrimiento de las familias desplazadas, sin duda, nos muestra de nuevo esa cara de la moneda donde destaca el perfil de Caín. Viejos, niños, mujeres, hombres de familia, son desalojados de sus tierras y casas, por un nacionalismo excluyente. Se habla de una limpieza étnica, a 

Descartar:
- URLs de las notas
- Correos electrónicos

In [30]:
tmp = re.sub(r"\S+\.html?\b", " ", tmp) # URLs
tmp = re.sub(r"\S+(@|\(a\))\S+", " ", tmp)  # correos electrónicos
tmp = re.sub(r"\baas\b", "a las", tmp) # errata "aas" -> "a las"
print(tmp[:1000])

 
Epigrama
JORGE MANSILLA TORRES
La diputada Raquel Sevilla, dueña de una violencia verbal envidiable, camina con un collarín ortopédico.
Y no faltó el periodista
que con lenguaje letal,
dijo que la asambleísta
¡mejor se ponga un bozal..!
 
Liquidar a Milosevic
Tranquilizar la Mala Conciencia
JORGE RUIZ DUEÑAS
LOS voceros de la Casa Blanca y de la OTAN se han referido, a la fórmula de los bombardeos sobre las ciudades yugoslavas, como el castigo necesario para liquidar a Slobodan Milosevic. Incluso, se dice, el conflicto actual es el precio de no haber actuado antes en el desmembramiento de la Yugoslavia de Tito. El sufrimiento de las familias desplazadas, sin duda, nos muestra de nuevo esa cara de la moneda donde destaca el perfil de Caín. Viejos, niños, mujeres, hombres de familia, son desalojados de sus tierras y casas, por un nacionalismo excluyente. Se habla de una limpieza étnica, a la manera de lo ocurrido antes, pero es inocultable la violencia de todos los bandos y el registro

Cargar la lista de stopwords en español de ```nltk.corpus```

In [31]:
stopwords_es = set(stopwords.words("spanish"))
print(stopwords_es)

{'estábamos', 'eres', 'serás', 'tenga', 'algunos', 'fuésemos', 'tendrán', 'suyo', 'otras', 'tuvimos', 'estuviese', 'estuviésemos', 'ellos', 'tuve', 'sería', 'hubimos', 'y', 'hubiste', 'hubiéramos', 'e', 'estuvimos', 'hubiésemos', 'estaré', 'esto', 'estás', 'su', 'estaremos', 'tenemos', 'tuya', 'hubieseis', 'esa', 'suyos', 'hubiese', 'mí', 'míos', 'estados', 'había', 'tuvieras', 'estaréis', 'el', 'nosotros', 'suya', 'a', 'tened', 'porque', 'seréis', 'algo', 'ante', 'hubieses', 'tuyas', 'hay', 'tenido', 'tú', 'seáis', 'seríamos', 'fuimos', 'yo', 'no', 'estando', 'hubierais', 'hubieras', 'tengas', 'fui', 'sentido', 'fueras', 'donde', 'nosotras', 'sea', 'pero', 'os', 'habría', 'hasta', 'tengáis', 'tuvo', 'seríais', 'mis', 'con', 'hubo', 'eran', 'estuvisteis', 'hubieran', 'entre', 'tendrás', 'antes', 'fuerais', 'como', 'estoy', 'suyas', 'esté', 'tenida', 'sentida', 'vuestra', 'éramos', 'tuvieron', 'seas', 'mías', 'sois', 'seremos', 'sean', 'estemos', 'mucho', 'habido', 'tuviste', 'hubieron'

Tokenización y lematización con spaCy.

Se lematiza el texto completo (con mayúsculas y puntuación) para que spaCy use el contexto de cada oración. Después se pasa a minúsculas y se filtran stopwords, signos de puntuación, números y tokens alfanuméricos.

In [32]:
nlp = spacy.load("es_core_news_md")
doc = nlp(tmp)

normalized_text = []
for token in doc:
    if not token.is_alpha or token.text.lower() in stopwords_es:
        continue
    # Verbos con pronombre pegado ("convertirse") dan el lema "convertir él": nos quedamos con el verbo
    lemma = token.lemma_.lower().split()[0]
    if lemma not in stopwords_es:
        normalized_text.append(lemma)

### 1. Texto normalizado

In [33]:
print(normalized_text[:100])

['epigrama', 'jorge', 'mansilla', 'torr', 'diputada', 'raquel', 'sevilla', 'dueña', 'violencia', 'verbal', 'envidiable', 'caminar', 'collarín', 'ortopédico', 'faltar', 'periodista', 'lenguaje', 'letal', 'decir', 'asambleísta', 'mejor', 'poner', 'bozal', 'liquidar', 'milosevic', 'tranquilizar', 'mala', 'conciencia', 'jorge', 'ruiz', 'dueñas', 'vocero', 'casa', 'blanca', 'otan', 'referir', 'fórmula', 'bombardeo', 'ciudad', 'yugoslavo', 'castigo', 'necesario', 'liquidar', 'slobodan', 'milosevic', 'incluso', 'decir', 'conflicto', 'actual', 'precio', 'haber', 'actuar', 'desmembramiento', 'yugoslavia', 'tito', 'sufrimiento', 'familia', 'desplazado', 'duda', 'mostrar', 'nuevo', 'cara', 'moneda', 'destacar', 'perfil', 'caín', 'viejos', 'niño', 'mujer', 'hombre', 'familia', 'desalojar', 'tierra', 'casa', 'nacionalismo', 'excluyente', 'hablar', 'limpieza', 'étnico', 'manera', 'ocurrido', 'inocultable', 'violencia', 'bando', 'registro', 'atrocidad', 'religión', 'diferencia', 'ocasión', 'superfici

Guardamos el texto normalizado en un txt por cualquier cosa

In [34]:
with open("tokens.txt", "w+") as file:
    file.write(" ".join(normalized_text))

Vocabulario y conteo de palabras.

In [35]:
locale.setlocale(locale.LC_COLLATE, "es_MX.UTF-8")

word_count = Counter(normalized_text)
word_count = dict(sorted(word_count.items(), key=lambda par: locale.strxfrm(par[0])))
print(word_count)

{'abajo': 3, 'abanderado': 3, 'abandonado': 2, 'abandonar': 2, 'abandono': 1, 'abastecer': 1, 'abatir': 2, 'abdicar': 1, 'aber': 1, 'aberrante': 1, 'abiertamente': 1, 'abierto': 6, 'abigeato': 2, 'abm': 1, 'abogado': 8, 'abogados': 2, 'abogar': 1, 'aboliar': 1, 'abolir': 1, 'abonar': 1, 'abono': 1, 'abordar': 1, 'abrazar': 2, 'abrego': 1, 'abril': 12, 'abrir': 6, 'absolutismo': 1, 'absoluto': 3, 'abstener': 1, 'abundancia': 1, 'abundar': 5, 'acabar': 8, 'academia': 1, 'académico': 21, 'académicos': 1, 'acahualtepec': 1, 'acalorado': 1, 'acapulco': 2, 'acaso': 1, 'acatar': 2, 'acceder': 3, 'acceso': 12, 'accesos': 1, 'accidentar': 1, 'accidente': 2, 'acción': 39, 'accionario': 4, 'acelerar': 1, 'aceptable': 3, 'aceptación': 2, 'aceptar': 15, 'acepten': 1, 'acepter': 1, 'acerca': 2, 'acercamiento': 1, 'acercar': 5, 'acercáronsele': 1, 'acertado': 1, 'aclarar': 15, 'acometer': 1, 'acompañado': 6, 'acompañar': 4, 'acontecer': 1, 'acontecimiento': 3, 'acordado': 2, 'acordar': 4, 'acosta': 1

In [36]:
vocabulary = word_count.keys()
vocabulary = list(vocabulary)
print(vocabulary)

['abajo', 'abanderado', 'abandonado', 'abandonar', 'abandono', 'abastecer', 'abatir', 'abdicar', 'aber', 'aberrante', 'abiertamente', 'abierto', 'abigeato', 'abm', 'abogado', 'abogados', 'abogar', 'aboliar', 'abolir', 'abonar', 'abono', 'abordar', 'abrazar', 'abrego', 'abril', 'abrir', 'absolutismo', 'absoluto', 'abstener', 'abundancia', 'abundar', 'acabar', 'academia', 'académico', 'académicos', 'acahualtepec', 'acalorado', 'acapulco', 'acaso', 'acatar', 'acceder', 'acceso', 'accesos', 'accidentar', 'accidente', 'acción', 'accionario', 'acelerar', 'aceptable', 'aceptación', 'aceptar', 'acepten', 'acepter', 'acerca', 'acercamiento', 'acercar', 'acercáronsele', 'acertado', 'aclarar', 'acometer', 'acompañado', 'acompañar', 'acontecer', 'acontecimiento', 'acordado', 'acordar', 'acosta', 'acotada', 'acotar', 'acrecentar', 'acreditar', 'acreditir', 'acreedor', 'acta', 'acteal', 'actitud', 'activar', 'actividad', 'activo', 'acto', 'actor', 'actuación', 'actual', 'actualidad', 'actualización'

Contexto de cada palabra del vocabulario

In [37]:
window = 8
contexts = defaultdict(list)

for i, word in enumerate(normalized_text):
    left = normalized_text[max(0, i - window // 2) : i]
    right = normalized_text[i + 1 : i + 1 + window // 2]
    contexts[word].extend(left + right)

for word in vocabulary[:10]:
    print(word, contexts[word])

abajo ['usted', 'señor', 'borges', 'allá', 'cerquita', 'aldea', 'poder', 'ver', 'señor', 'borges', 'guitarrero', 'allá', 'hombre', 'guitarra', 'primero', 'aldeano', 'alcanzar', 'nivel', 'construcción', 'todavía', 'participación', 'economía', 'venir', 'menos']
abanderado ['cen', 'mismo', 'día', 'designarían', 'capital', 'antonio', 'garza', 'consejo', 'institucional', 'votar', 'directamente', 'elegir', 'presidencial', 'mismo', 'fecha', 'determinado', 'julio', 'definir', 'procedimiento', 'elección', 'agosto', 'iniciar', 'campaña', 'aspirante']
abandonado ['recurso', 'presupuestal', 'denunciar', 'prácticamente', 'institución', 'público', 'entidad', 'destinar', 'distribución', 'recurso', 'presupuestal', 'prácticamente', 'institución', 'público', 'entidad', 'destinar']
abandonar ['tubo', 'luz', 'pensamiento', 'recibir', 'oficio', 'silencio', 'referir', 'priísta', 'ejército', 'liberación', 'kosovo', 'deber', 'actividad', 'elección', 'libre', 'conformación']
abandono ['sino', 'grave', 'condici

### 2. La matriz término-documento (TF)

$$\text{TF}(\text{raw frequency}) = \text{raw frequency} $$

In [38]:
index = {word: j for j, word in enumerate(vocabulary)}
rawfreq_term_doc_matrix = np.zeros((len(vocabulary), len(vocabulary)), dtype=np.int32)

# Las filas son las palabras y las columnas son los contextos de las palabras
for j, target in enumerate(vocabulary):
    for word, count in Counter(contexts[target]).items():
        rawfreq_term_doc_matrix[index[word], j] = count

# Las palabras seleccionadas son aquellas cuyo valor no es cero para poder ver cómo luce la matriz más o menos
sample_words = ["partido", "político", "pri", "candidato", "gobierno", "elección", "presidente", "zedillo"]
df = pd.DataFrame(rawfreq_term_doc_matrix, index=vocabulary, columns=vocabulary)
df = df.rename_axis(index="palabra", columns="contexto")
display(df.loc[sample_words, sample_words])

contexto,partido,político,pri,candidato,gobierno,elección,presidente,zedillo
palabra,,,,,,,,
partido,4,16,4,3,1,1,2,1
político,16,6,1,0,1,1,1,0
pri,4,1,0,4,3,1,2,1
candidato,3,0,4,0,6,7,0,1
gobierno,1,1,3,6,0,4,1,4
elección,1,1,1,7,4,0,1,2
presidente,2,1,2,0,1,1,0,10
zedillo,1,0,1,1,4,2,10,0


$$ \text{TF}(\text{raw frequency}) = \frac{\text{raw frequency}}{\sum{\text{raw frequencies}}} $$

In [39]:
relfreq_term_doc_matrix = rawfreq_term_doc_matrix / rawfreq_term_doc_matrix.sum(axis=0)

df = pd.DataFrame(relfreq_term_doc_matrix, index=vocabulary, columns=vocabulary)
df = df.rename_axis(index="palabra", columns="contexto")
display(df.loc[sample_words, sample_words])

contexto,partido,político,pri,candidato,gobierno,elección,presidente,zedillo
palabra,,,,,,,,
partido,0.007246,0.027778,0.010204,0.011364,0.001389,0.003676,0.003731,0.006579
político,0.028986,0.010417,0.002551,0.000000,0.001389,0.003676,0.001866,0.000000
pri,0.007246,0.001736,0.000000,0.015152,0.004167,0.003676,0.003731,0.006579
candidato,0.005435,0.000000,0.010204,0.000000,0.008333,0.025735,0.000000,0.006579
gobierno,0.001812,0.001736,0.007653,0.022727,0.000000,0.014706,0.001866,0.026316
elección,0.001812,0.001736,0.002551,0.026515,0.005556,0.000000,0.001866,0.013158
presidente,0.003623,0.001736,0.005102,0.000000,0.001389,0.003676,0.000000,0.065789
zedillo,0.001812,0.000000,0.002551,0.003788,0.005556,0.007353,0.018657,0.000000


$$ \text{TF}(\text{raw frequency}) = \begin{cases} 0 \text{, raw fequency} = 0 \\ 1 \text{, raw frequency} \ge 1  \end{cases}  $$

In [40]:
binary_term_doc_matrix = (rawfreq_term_doc_matrix > 0).astype(int)

df = pd.DataFrame(binary_term_doc_matrix, index=vocabulary, columns=vocabulary)
df = df.rename_axis(index="palabra", columns="contexto")
display(df.loc[sample_words, sample_words])

contexto,partido,político,pri,candidato,gobierno,elección,presidente,zedillo
palabra,,,,,,,,
partido,1,1,1,1,1,1,1,1
político,1,1,1,0,1,1,1,0
pri,1,1,0,1,1,1,1,1
candidato,1,0,1,0,1,1,0,1
gobierno,1,1,1,1,0,1,1,1
elección,1,1,1,1,1,0,1,1
presidente,1,1,1,0,1,1,0,1
zedillo,1,0,1,1,1,1,1,0


$$ \text{TF}(\text{raw frequency}) = \log{(1 + \text{raw frequency})} $$

In [41]:
log_term_doc_matrix = np.log1p(rawfreq_term_doc_matrix)

df = pd.DataFrame(log_term_doc_matrix, index=vocabulary, columns=vocabulary)
df = df.rename_axis(index="palabra", columns="contexto")
display(df.loc[sample_words, sample_words])

contexto,partido,político,pri,candidato,gobierno,elección,presidente,zedillo
palabra,,,,,,,,
partido,1.609438,2.833213,1.609438,1.386294,0.693147,0.693147,1.098612,0.693147
político,2.833213,1.945910,0.693147,0.000000,0.693147,0.693147,0.693147,0.000000
pri,1.609438,0.693147,0.000000,1.609438,1.386294,0.693147,1.098612,0.693147
candidato,1.386294,0.000000,1.609438,0.000000,1.945910,2.079442,0.000000,0.693147
gobierno,0.693147,0.693147,1.386294,1.945910,0.000000,1.609438,0.693147,1.609438
elección,0.693147,0.693147,0.693147,2.079442,1.609438,0.000000,0.693147,1.098612
presidente,1.098612,0.693147,1.098612,0.000000,0.693147,0.693147,0.000000,2.397895
zedillo,0.693147,0.000000,0.693147,0.693147,1.609438,1.098612,2.397895,0.000000


 $$ \text{TF}(\text{raw frequency}) = \frac{(k + 1) \cdot \text{raw frequency}}{\text{raw frequency} + k \cdot \left(1 - b + b \cdot \frac{|contexto|}{\text{avgdl}}\right)} $$

In [42]:
k, b = 1.2, 0.75
contexts_len = rawfreq_term_doc_matrix.sum(axis=0)
avgdl = contexts_len.mean()
length_norm = k * (1 - b + b * contexts_len / avgdl)
BM25_term_doc_matrix = ((k + 1) * rawfreq_term_doc_matrix) / (rawfreq_term_doc_matrix + length_norm)
BM25_term_doc_matrix = BM25_term_doc_matrix / BM25_term_doc_matrix.sum(axis=0)

df = pd.DataFrame(BM25_term_doc_matrix, index=vocabulary, columns=vocabulary)
df = df.rename_axis(index="palabra", columns="contexto")
display(df.loc[sample_words, sample_words])

contexto,partido,político,pri,candidato,gobierno,elección,presidente,zedillo
palabra,,,,,,,,
partido,0.006502,0.015645,0.008618,0.010134,0.001460,0.003929,0.003688,0.007141
político,0.016130,0.008502,0.002686,0.000000,0.001460,0.003929,0.001958,0.000000
pri,0.006502,0.001827,0.000000,0.012349,0.004003,0.003929,0.003688,0.007141
candidato,0.005138,0.000000,0.008618,0.000000,0.007091,0.016383,0.000000,0.007141
gobierno,0.001919,0.001827,0.006920,0.015802,0.000000,0.011734,0.001958,0.018499
elección,0.001919,0.001827,0.002686,0.017175,0.005117,0.000000,0.001958,0.012090
presidente,0.003620,0.001827,0.004964,0.000000,0.001460,0.003929,0.000000,0.027127
zedillo,0.001919,0.000000,0.002686,0.004162,0.005117,0.007060,0.012574,0.000000


### 3. El vector IDF

$$ \text{IDF}(w_i) = \log{(\frac{\text{len(vocabulario)} + 1}{\text{\# de contextos en los que aparece  }w_i})} $$

In [43]:
M = len(vocabulary)
doc_freq = binary_term_doc_matrix.sum(axis=1)
idf = np.log((M + 1) / doc_freq)

idf_df = pd.DataFrame({"IDF": idf}, index=vocabulary).rename_axis(index="palabra")
display(idf_df)

,IDF
palabra,
abajo,5.727003
abanderado,5.636031
abandonado,6.574301
abandonar,5.998937
abandono,6.692084
...,...
zona,4.156405
zoque,6.692084
zozobra,6.692084


el mismo vector IDF sirve para cualquier matriz TF.

Ahora calculamos la matriz TF-IDF para cada variante de TF.

$$
\text{TF-IDF} =
\begin{bmatrix} \text{TF}_{c_1} \odot \text{IDF} & \cdots & \text{TF}_{c_M} \odot \text{IDF} \end{bmatrix} =
\begin{bmatrix}
\text{TF}(w_1, c_1) \cdot \text{IDF}(w_1) & \cdots & \text{TF}(w_1, c_M) \cdot \text{IDF}(w_1) \\
\vdots & \ddots & \vdots \\
\text{TF}(w_M, c_1) \cdot \text{IDF}(w_M) & \cdots & \text{TF}(w_M, c_M) \cdot \text{IDF}(w_M)
\end{bmatrix}
$$

In [44]:
tf_idf_matrices = [
    ("rawfreq", rawfreq_term_doc_matrix * idf[:, np.newaxis]),
    ("relfreq", relfreq_term_doc_matrix * idf[:, np.newaxis]),
    ("binary", binary_term_doc_matrix * idf[:, np.newaxis]),
    ("log", log_term_doc_matrix * idf[:, np.newaxis]),
    ("BM25", BM25_term_doc_matrix * idf[:, np.newaxis]),
]

### 4. Definición de "similitud"

Similitud coseno:

$$ \text{Similitud}(w_1, w_2) = \cos{(\theta)} = \frac{contexto_{w_1} \cdot contexto_{w_2}}{||contexto_{w_1}||_2 \cdot ||contexto_{w_2}||_2} $$

In [45]:
def cosine_similarity(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))

Distancia Euclidiana:

$$ \text{Similitud}(w_1, w_2) = || contexto_{w_1} - contexto_{w_2} ||_2 $$

In [46]:
def euclidean_distance(u, v):
      return np.linalg.norm(u - v)

Producto punto:

$$ \text{Similitud}(w_1, w_2) = contexto_{w_1} \cdot contexto_{w_2} = \sum_i{(contexto_{w_1})_i \cdot (contexto_{w_2})_i} $$

In [47]:
def dot_product(u, v):
      return u @ v

### 5. Palabras más similares con "partido" en cada matriz TF-IDF

In [48]:
query_word = "partido"
query_index = index[query_word]

#### 5.1 Similitud coseno

In [49]:
cosine_table = pd.DataFrame(index=range(1, 11))

for name, tf_idf in tf_idf_matrices:
    similarities = np.zeros(len(vocabulary))
    for m in range(len(vocabulary)):
        similarities[m] = cosine_similarity(tf_idf[:, query_index], tf_idf[:, m])

    order = np.argsort(similarities)[::-1]  # de mayor a menor similitud
    order = order[order != query_index][:10]  # sin la propia palabra
    cosine_table[name] = [f"{vocabulary[m]} ({similarities[m]:.3f})" for m in order]

display(cosine_table)

,rawfreq,relfreq,binary,log,BM25
1,político (0.337),político (0.337),político (0.231),político (0.286),político (0.319)
2,pri (0.272),pri (0.272),pri (0.191),pri (0.244),pri (0.268)
3,adversario (0.259),adversario (0.259),considerar (0.179),poder (0.213),poder (0.243)
4,hacer (0.251),hacer (0.251),perredista (0.174),considerar (0.208),hacer (0.235)
5,poder (0.247),poder (0.247),nacional (0.169),candidato (0.202),gobierno (0.227)
6,decir (0.236),decir (0.236),nominación (0.168),gobierno (0.201),decir (0.223)
7,gobierno (0.228),gobierno (0.228),candidato (0.168),hacer (0.201),considerar (0.222)
8,considerar (0.228),considerar (0.228),poder (0.167),nacional (0.198),nacional (0.211)
9,proceso (0.225),proceso (0.225),prd (0.163),nominación (0.193),proceso (0.211)
10,ver (0.222),ver (0.222),hacer (0.159),decir (0.190),candidato (0.210)


#### 5.2 Distancia euclidiana

In [50]:
euclidean_table = pd.DataFrame(index=range(1, 11))

for name, tf_idf in tf_idf_matrices:
    distances = np.zeros(len(vocabulary))
    for m in range(len(vocabulary)):
        distances[m] = euclidean_distance(tf_idf[:, query_index], tf_idf[:, m])

    order = np.argsort(distances)  # de menor a mayor distancia
    order = order[order != query_index][:10]  # sin la propia palabra
    euclidean_table[name] = [f"{vocabulary[m]} ({distances[m]:.3f})" for m in order]

display(euclidean_table)

,rawfreq,relfreq,binary,log,BM25
1,adversario (134.331),poder (0.269),adversario (99.504),adversario (77.706),poder (0.261)
2,porfirio (135.870),hacer (0.270),nominación (99.523),nominación (77.763),hacer (0.266)
3,asentar (135.905),decir (0.272),unilateralmente (99.525),asentar (78.226),decir (0.267)
4,nominación (135.934),méxico (0.283),conservir (99.526),unilateralmente (78.243),méxico (0.277)
5,claridad (136.023),ser (0.285),invocar (99.532),amenacir (78.247),ser (0.278)
6,mérida (136.300),político (0.287),matar (99.541),balanza (78.249),político (0.281)
7,revolución (136.490),país (0.293),izquierda (99.543),revolución (78.278),gobierno (0.286)
8,perredista (136.500),año (0.296),guanajuato (99.573),democrática (78.310),país (0.289)
9,dependiente (136.540),si (0.296),amenacir (99.640),pvem (78.339),año (0.289)
10,democrática (136.642),gobierno (0.297),agrupamiento (99.646),guanajuato (78.347),si (0.291)


#### 5.3 Producto punto

In [51]:
dot_table = pd.DataFrame(index=range(1, 11))

for name, tf_idf in tf_idf_matrices:
    products = np.zeros(len(vocabulary))
    for m in range(len(vocabulary)):
        products[m] = dot_product(tf_idf[:, query_index], tf_idf[:, m])

    order = np.argsort(products)[::-1]  # de mayor a menor producto punto
    order = order[order != query_index][:10]  # sin la propia palabra
    dot_table[name] = [f"{vocabulary[m]} ({products[m]:.4g})" for m in order]

display(dot_table)

,rawfreq,relfreq,binary,log,BM25
1,poder (8149),dependiente (0.0737),poder (2555),poder (2116),balanza (0.06426)
2,hacer (6908),patrocinan (0.07147),político (2377),político (1826),unilateralmente (0.06317)
3,político (6637),firmeza (0.0706),hacer (2210),hacer (1768),dependiente (0.06012)
4,decir (6564),afuera (0.06802),decir (2157),decir (1731),guanajuato (0.06005)
5,gobierno (5154),unilateralmente (0.06657),nacional (1856),ser (1413),recinto (0.058)
6,ser (5137),agremiado (0.06619),méxico (1814),gobierno (1404),presidenciales (0.0572)
7,méxico (5083),opuesta (0.066),ser (1771),méxico (1384),vasco (0.05694)
8,nacional (4554),aprestar (0.06555),gobierno (1747),nacional (1378),aparente (0.05569)
9,año (4489),balanza (0.06432),pri (1615),pri (1286),secta (0.05558)
10,pri (4263),adversario (0.06343),considerar (1583),año (1234),agremiado (0.0549)
